In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [2]:
import pickle
import numpy as np
from tqdm.auto import tqdm
from functools import partial

import jax
import jax.numpy as jnp

# astropy + gala just to get the initial 6D phase-space point
import astropy.units as u
import astropy.coordinates as coord
import gala.dynamics as gd

# main dynamics library
import streamsculptor as js

import matplotlib.pyplot as plt

In [3]:
# set unit system (kpc, Myr, Msun, rad)
#usys = js.UnitSystem(u.kpc, u.Myr, u.Msun, u.radian)

# GD-1 progenitor coordinates from Webb & Bovy (2019)
# starting in ICRS and then converting to Galactocentric
gd1_c = coord.SkyCoord(
    ra=148.9363998668805 * u.degree,
    dec=36.15980426805254 * u.degree,
    distance=7.555339165941959 * u.kpc,
    pm_ra_cosdec=-5.332929760383195 * u.mas/u.yr,
    pm_dec=-12.198914465325117 * u.mas/u.yr,
    radial_velocity=6.944006091929623 * u.km/u.s,
    frame="icrs"
)

rep = gd1_c.transform_to(coord.Galactocentric).data
gd1_w0 = gd.PhaseSpacePosition(rep)

# flatten into a 6D vector for JAX (x,y,z,vx,vy,vz)
GD1_W0 = jnp.hstack([
    gd1_w0.pos.xyz.T.value,
    gd1_w0.vel.d_xyz.to(u.kpc/u.Myr).value
])

In [4]:
# simple Milky Way model: Miyamoto-Nagai disk + NFW halo
disk = js.MiyamotoNagaiDisk(
    m=6.8e10,
    a=3.0,
    b=0.28,
    units=usys
)

halo = js.NFWPotential(
    m=8e11,
    r_s=16.0,
    units=usys
)

bulge = js.HernquistPotential(
    m=4e9, 
    c=1.0, 
    units=usys)

MW_POTENTIAL_GLOBAL = js.Potential_Combine(
    [disk, halo, bulge],
    units=usys
)

# integrate GD-1 orbit backwards 2 Gyr
def integrate_gd1_backwards(potential, w0):
    ts = jnp.linspace(0.0, -2000.0, 1000)
    orbit  = potential.orbit_integrator_run(w0, 0.0, -2000.0, ts)
    return orbit[:-1, :]

# compute once and reuse for all streams
GD1_ORBIT = integrate_gd1_backwards(
    MW_POTENTIAL_GLOBAL, GD1_W0
)
print(f"Orbit initialized. Shape: {GD1_ORBIT.shape}")

AttributeError: module 'streamsculptor' has no attribute 'MiyamotoNagaiDisk'

SyntaxError: invalid syntax (3925404626.py, line 1)